# Gold Annotation — Stratified Sample Builder

Produces a CSV for manual linguistic verification of a Billex bilingual
lexicon. Uses **stratified sampling** so every known risk class is
represented in the annotation set, giving unbiased precision/recall
estimates per error class.

## Label schema

Each sampled entry gets exactly one of these labels:

| Label | Meaning |
|---|---|
| `correct` | Indonesian headword AND regional translation both correct |
| `ocr_error` | One or both sides contain OCR/typography corruption (wrong characters, garbled word) |
| `extraction_boundary_error` | The entry boundaries were drawn incorrectly upstream (e.g. previous entry's content bled into this one) |
| `ambiguous` | Annotator cannot decide without consulting additional sources |

Optional columns for finer-grained analysis:
- `error_side` — `ind`, `reg`, or `both` (filled only when label ≠ correct)
- `correction` — the annotator's suggested fix (free-text, optional)
- `notes` — free-text reasoning

## Sample size

For dict 4 (Jambi A–K) with ~1,900 entries and ~28% known anomaly rate,
**250 stratified entries** gives ±4% precision at 95% CI on each error
class. This is 4–6 hours of annotation for one trained linguist.

## 1. Inputs

Point this at your Billex audit CSV (either the original
`4_Billex.csv` or the cleaned `4_Billex_audit.csv` from the batch fix).

For additional context, the script also optionally reads:
- the preprocessing output (for `is_anomali` and `page` metadata)
- the parallel corpus (for example-sentence context)

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import ast

# === CONFIGURE THESE ===
DICT_ID = "4"  # which dictionary to annotate
BILLEX_PATH = Path("../Ekstraksi/9. Bilingual Lexicon - Fixed") / f"{DICT_ID}_Billex_audit.csv"
# Fallback to original if audit doesn't exist:
if not BILLEX_PATH.exists():
    BILLEX_PATH = Path("../Ekstraksi/9. Bilingual Lexicon") / f"{DICT_ID}_Billex.csv"

# Optional enrichment sources (set to None if not available)
PREP_PATH = Path("../Ekstraksi/4. Preprocessing - Remove Anomali Entri") / f"{DICT_ID}_Replace-categorizeAnomali-WithMainEntry-splitted.csv"
PARCOR_PATH = Path("../Ekstraksi/11. Parallel Corpus") / f"{DICT_ID}_Parcor.csv"

OUT_DIR = Path("../evaluation/gold_entries")
OUT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 42
TOTAL_SAMPLE_SIZE = 250

print(f"Billex:  {BILLEX_PATH}  exists={BILLEX_PATH.exists()}")
print(f"Prep:    {PREP_PATH}    exists={PREP_PATH.exists()}")
print(f"Parcor:  {PARCOR_PATH}  exists={PARCOR_PATH.exists()}")

Billex:  ..\Ekstraksi\9. Bilingual Lexicon - Fixed\4_Billex_audit.csv  exists=True
Prep:    ..\Ekstraksi\4. Preprocessing - Remove Anomali Entri\4_Replace-categorizeAnomali-WithMainEntry-splitted.csv    exists=False
Parcor:  ..\Ekstraksi\11. Parallel Corpus\4_Parcor.csv  exists=True


In [2]:
billex = pd.read_csv(BILLEX_PATH)
print(f"Loaded {len(billex)} Billex entries")
print(f"Columns: {list(billex.columns)}")
billex.head()

Loaded 1894 Billex entries
Columns: ['kata_asal', 'kata_tujuan', 'makna', 'kata_asal_original', 'kata_asal_cleaned', 'cleanup_tag', 'direction', 'direction_known']


,kata_asal,kata_tujuan,makna,kata_asal_original,kata_asal_cleaned,cleanup_tag,direction,direction_known
0,abai,['lale'],[],abai,abai,ok,1,True
1,pengabdian,['pengabdian'],[],pengabdian,pengabdian,ok,1,True
2,abnormal,['manyimpang'],[],abnormal,abnormal,ok,1,True
3,absen,['absen'],[],absen,absen,ok,1,True
4,absensi absènsil,['ketidakhadighan'],[],absensi absènsil,absensi,phonetic_stripped,1,True


## 2. Compute risk signals on every entry

Each entry gets four binary risk flags. Stratification uses these flags to
allocate sample slots.

In [3]:
import re

def parse_list(s):
    try:
        v = ast.literal_eval(s) if isinstance(s, str) else []
        return v if isinstance(v, list) else []
    except Exception:
        return []

# Identify which column is Indonesian vs regional
if 'direction' in billex.columns:
    direction = int(billex['direction'].iloc[0])
else:
    direction = 1  # assume Indonesian->Regional if not specified
ind_col = 'kata_asal' if direction == 1 else 'kata_tujuan'
reg_col = 'kata_tujuan' if direction == 1 else 'kata_asal'

# First translation (regional side is usually a list)
def first_of(v):
    lst = parse_list(v) if isinstance(v, str) and v.startswith('[') else None
    if lst:
        return str(lst[0]).strip()
    return str(v).strip()

ind_series = billex[ind_col].astype(str).str.strip()
reg_series = billex[reg_col].apply(first_of) if direction == 1 else billex[reg_col].astype(str).str.strip()
# For dir=0, kata_tujuan (Indonesian gloss) is a list; swap the logic:
if direction == 0:
    ind_series = billex[ind_col].apply(first_of)
    reg_series = billex[reg_col].astype(str).str.strip()

flags = pd.DataFrame({
    'row_id': range(len(billex)),
    'indonesian': ind_series,
    'regional': reg_series,
})

# Risk signal 1: multi-word Indonesian headword (extraction noise)
flags['is_multiword_ind'] = flags['indonesian'].str.contains(r'\s', na=False)

# Risk signal 2: identical source = target (either real loan or unhandled gap)
flags['is_identical'] = (flags['indonesian'].str.lower() == flags['regional'].str.lower()) & (flags['indonesian'] != '')

# Risk signal 3: empty regional side
flags['is_empty_reg'] = flags['regional'].isin(['', '[]', 'nan'])

# Risk signal 4: contains suspicious characters (digits, Latin-1, triple-repeat)
def has_noise_chars(s):
    if not isinstance(s, str): return False
    return bool(re.search(r'\d|(.)\1\1|[À-ÿ]', s))
flags['has_noise_chars'] = flags['regional'].apply(has_noise_chars) | flags['indonesian'].apply(has_noise_chars)

# Summary
print("Risk flag counts:")
for col in ['is_multiword_ind', 'is_identical', 'is_empty_reg', 'has_noise_chars']:
    print(f"  {col}: {flags[col].sum()}")
flags.head()

Risk flag counts:
  is_multiword_ind: 226
  is_identical: 540
  is_empty_reg: 28
  has_noise_chars: 103


,row_id,indonesian,regional,is_multiword_ind,is_identical,is_empty_reg,has_noise_chars
0,0,abai,lale,False,False,False,False
1,1,pengabdian,pengabdian,False,True,False,False
2,2,abnormal,manyimpang,False,False,False,False
3,3,absen,absen,False,True,False,False
4,4,absensi absènsil,ketidakhadighan,True,False,False,True


## 3. Stratified sampling

We allocate the 250-entry budget across four strata:

| Stratum | Description | Target size |
|---|---|---|
| A | High-risk: ANY risk flag set | 150 |
| B | Clean baseline: NO risk flags | 80 |
| C | Edge cases: empty regional side | all (≤20) |
| D | Edge cases: multi-word regional | all (≤20) |

Strata C and D are usually small, so we take all of them. Strata A and B
are sampled uniformly at random within the group.

The clean baseline (B) is critical: without it you can only report on
errors you already suspected. B gives an unbiased estimate of the
lexicon's true error rate.

In [4]:
rng = np.random.default_rng(RANDOM_SEED)

# Define strata
any_risk = flags['is_multiword_ind'] | flags['is_identical'] | flags['is_empty_reg'] | flags['has_noise_chars']
stratum_A = flags[any_risk].copy()
stratum_B = flags[~any_risk].copy()
stratum_C = flags[flags['is_empty_reg']].copy()
stratum_D_mask = flags['regional'].astype(str).str.contains(r'\s', na=False)
stratum_D = flags[stratum_D_mask].copy()

print(f"Stratum A (any risk):       {len(stratum_A)}")
print(f"Stratum B (clean baseline): {len(stratum_B)}")
print(f"Stratum C (empty reg):      {len(stratum_C)}")
print(f"Stratum D (multi-word reg): {len(stratum_D)}")

def sample_n(df, n, label):
    if len(df) == 0:
        return pd.DataFrame()
    n = min(n, len(df))
    sampled = df.sample(n=n, random_state=RANDOM_SEED).copy()
    sampled['stratum'] = label
    return sampled

sample_A = sample_n(stratum_A, 150, 'A_high_risk')
sample_B = sample_n(stratum_B, 80, 'B_clean_baseline')
sample_C = sample_n(stratum_C, min(20, len(stratum_C)), 'C_empty_reg')
sample_D = sample_n(stratum_D, min(20, len(stratum_D)), 'D_multiword_reg')

# Combine and dedupe by row_id (C/D may overlap with A)
sample = pd.concat([sample_A, sample_B, sample_C, sample_D])
sample = sample.drop_duplicates(subset='row_id', keep='first').reset_index(drop=True)

print(f"\nFinal sample size: {len(sample)}")
print(f"Per stratum:\n{sample['stratum'].value_counts()}")

Stratum A (any risk):       776
Stratum B (clean baseline): 1118
Stratum C (empty reg):      28
Stratum D (multi-word reg): 50

Final sample size: 264
Per stratum:
stratum
A_high_risk         150
B_clean_baseline     80
D_multiword_reg      18
C_empty_reg          16
Name: count, dtype: int64


## 4. Enrich with context

For each sampled entry, pull in example sentences and the preprocessing
metadata (page number, anomaly flag). The annotator uses these to verify
against the PDF without re-loading it per row.

In [5]:
enriched = sample.merge(
    billex.reset_index().rename(columns={'index': 'row_id'}),
    on='row_id', how='left', suffixes=('', '_billex')
)

# Pull page + is_anomali from preprocessing if available
if PREP_PATH.exists():
    prep = pd.read_csv(PREP_PATH)

    # Extract the headword from definisi_lema: it's the first whitespace-
    # separated token with dots removed (e.g. 'a.ba-a.ba' -> 'aba-aba').
    # This matches the normalisation done in notebook 13.
    def extract_key(definisi):
        if not isinstance(definisi, str):
            return ''
        first = definisi.strip().split()[0] if definisi.strip() else ''
        return first.replace('.', '').lower()

    prep_slim = prep[['definisi_lema', 'page', 'is_anomali', 'contoh_kalimat']].copy()
    prep_slim['lookup_key'] = prep_slim['definisi_lema'].apply(extract_key)
    # Keep first occurrence per key (same headword can appear on multiple pages)
    prep_slim = prep_slim.drop_duplicates('lookup_key', keep='first')

    enriched['lookup_key'] = enriched['indonesian'].astype(str).str.lower().str.replace(r'\s+', '', regex=True)
    enriched = enriched.merge(
        prep_slim[['lookup_key', 'page', 'is_anomali', 'contoh_kalimat']],
        on='lookup_key', how='left'
    ).drop(columns=['lookup_key'])

    matched = enriched['page'].notna().sum()
    print(f"Matched page/anomaly info for {matched}/{len(enriched)} entries ({matched/len(enriched):.1%})")
else:
    enriched['page'] = ''
    enriched['is_anomali'] = ''
    enriched['contoh_kalimat'] = ''

enriched.head()

,row_id,indonesian,regional,is_multiword_ind,is_identical,is_empty_reg,has_noise_chars,stratum,kata_asal,kata_tujuan,makna,kata_asal_original,kata_asal_cleaned,cleanup_tag,direction,direction_known,page,is_anomali,contoh_kalimat
0,984,gas,gas,False,True,False,False,A_high_risk,gas,['gas'],[],gas,gas,ok,1,True,,,
1,1739,komisi,komisi,False,True,False,False,A_high_risk,komisi,['komisi'],[],komisi,komisi,ok,1,True,,,
2,800,eksekusi,eksekusi,False,True,False,False,A_high_risk,eksekusi,['eksekusi'],[],eksekusi,eksekusi,ok,1,True,,,
3,1864,kuntum,kuntum,False,True,False,False,A_high_risk,kuntum,['kuntum'],[],kuntum,kuntum,ok,1,True,,,
4,858,ese lon iéselonl,eselon,True,False,False,True,A_high_risk,ese lon iéselonl,['eselon'],[],ese lon iéselonl,eselon,phonetic_stripped+syllable_rejoined,1,True,,,


## 5. Format for annotation

Produces a Google-Sheets-friendly CSV with:
- the entry to annotate (Indonesian + regional)
- context (page, stratum, example sentence)
- **empty label columns** for the annotator to fill

The linguist opens this in Google Sheets or Excel, skims the PDF for
unclear cases, and fills the `label`, `error_side`, `correction`, `notes`
columns.

In [6]:
gold = pd.DataFrame({
    'annotation_id': [f"{DICT_ID}_{i:04d}" for i in range(len(enriched))],
    'stratum': enriched['stratum'],
    'page': enriched['page'],
    'indonesian': enriched['indonesian'],
    'regional': enriched['regional'],
    'example_sentence': enriched['contoh_kalimat'],
    'pipeline_is_anomali': enriched['is_anomali'],
    # Empty columns for annotator to fill
    'label': '',                # correct / ocr_error / extraction_boundary_error / ambiguous
    'error_side': '',           # ind / reg / both  (only if label != correct)
    'correction': '',           # suggested fix
    'notes': '',
})

out_path = OUT_DIR / f"{DICT_ID}_gold_annotation_template.csv"
gold.to_csv(out_path, index=False)
print(f"Wrote {len(gold)} rows to: {out_path}")
gold.head(10)

Wrote 264 rows to: ..\evaluation\gold_entries\4_gold_annotation_template.csv


,annotation_id,stratum,page,indonesian,regional,example_sentence,pipeline_is_anomali,label,error_side,correction,notes
0,4_0000,A_high_risk,,gas,gas,,,,,,
1,4_0001,A_high_risk,,komisi,komisi,,,,,,
2,4_0002,A_high_risk,,eksekusi,eksekusi,,,,,,
3,4_0003,A_high_risk,,kuntum,kuntum,,,,,,
4,4_0004,A_high_risk,,ese lon iéselonl,eselon,,,,,,
5,4_0005,A_high_risk,,ingat,ingat,,,,,,
6,4_0006,A_high_risk,,kontinu,kontinu,,,,,,
7,4_0007,A_high_risk,,centong icéntong,centong,,,,,,
8,4_0008,A_high_risk,,batang,batang,,,,,,
9,4_0009,A_high_risk,,ideal iidéall,cocok,,,,,,


## 6. Annotator instructions (copy into the Google Sheet)

```
LABEL DEFINITIONS

  correct                    : both sides of the entry are accurate Indonesian
                               and regional-language forms
  ocr_error                  : at least one side contains corrupted characters
                               (wrong letters, missing letters, extra symbols)
                               that do not exist in the source dictionary
  extraction_boundary_error  : the entry's boundaries are wrong — content
                               from adjacent entries has bled in, or part
                               of this entry was lost to an adjacent one
  ambiguous                  : you cannot decide based on the PDF alone
                               (write why in `notes`)

ERROR_SIDE (fill only when label != correct):
  ind      : error is on the Indonesian side
  reg      : error is on the regional side
  both     : errors on both sides

CORRECTION (optional): write what the entry should have been, based on
  the PDF. Leave blank if not obvious.

PACING: aim for ~1 minute per entry. Flag the entry as `ambiguous` rather
than spending 5 minutes on one edge case.
```

## 7. Inter-annotator agreement (optional but recommended)

Duplicate 10% of the sample (25 rows) into a second sheet for a second
annotator. Compute Cohen's κ on the overlap. κ ≥ 0.7 is standard for
publication-quality annotation. Anything less means the label schema
needs sharpening before the full run.

In [7]:
# Create 10% overlap for IAA
iaa_subset = gold.sample(n=max(20, len(gold) // 10), random_state=RANDOM_SEED).copy()
iaa_subset.to_csv(OUT_DIR / f"{DICT_ID}_gold_annotation_IAA_subset.csv", index=False)
print(f"Wrote {len(iaa_subset)} rows for second-annotator IAA: "
      f"{OUT_DIR / f'{DICT_ID}_gold_annotation_IAA_subset.csv'}")

Wrote 26 rows for second-annotator IAA: ..\evaluation\gold_entries\4_gold_annotation_IAA_subset.csv


## 8. After annotation — compute metrics

Once the linguist returns the filled CSV, run this cell to compute:
- overall accuracy
- precision/recall of the pipeline's `is_anomali` flag vs. gold
- precision/recall of cleanup_tag (from the batch fix) vs. gold

This is how you'll produce the numbers for your thesis.

In [8]:
# Load the returned gold (rename this to match your annotated file)
annotated_path = OUT_DIR / f"{DICT_ID}_gold_annotation_COMPLETED.csv"

if annotated_path.exists():
    gold_done = pd.read_csv(annotated_path)
    valid = {'correct', 'ocr_error', 'extraction_boundary_error', 'ambiguous'}
    bad = gold_done[~gold_done['label'].isin(valid)]
    if len(bad) > 0:
        print(f"⚠ {len(bad)} rows have invalid labels — clean first:")
        print(bad[['annotation_id', 'label']].head())
    else:
        # Overall error rate
        err_rate = (gold_done['label'] != 'correct').mean()
        print(f"Overall error rate (excluding 'correct'): {err_rate:.1%}")
        print(f"\nLabel distribution:")
        print(gold_done['label'].value_counts())

        # Agreement with pipeline's is_anomali
        if 'pipeline_is_anomali' in gold_done.columns:
            pipeline_pos = gold_done['pipeline_is_anomali'].astype(str).isin(['1', '1.0', 'True'])
            gold_pos = gold_done['label'] != 'correct'
            tp = (pipeline_pos & gold_pos).sum()
            fp = (pipeline_pos & ~gold_pos).sum()
            fn = (~pipeline_pos & gold_pos).sum()
            if tp + fp > 0 and tp + fn > 0:
                prec = tp / (tp + fp)
                rec = tp / (tp + fn)
                print(f"\nPipeline is_anomali vs gold:")
                print(f"  Precision: {prec:.3f}")
                print(f"  Recall:    {rec:.3f}")
                print(f"  F1:        {2*prec*rec/(prec+rec):.3f}")
else:
    print(f"No completed annotation found at {annotated_path}")
    print("Run this cell after the linguist returns the filled CSV.")

No completed annotation found at ..\evaluation\gold_entries\4_gold_annotation_COMPLETED.csv
Run this cell after the linguist returns the filled CSV.
